<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Árboles de Decisión: el Juego de las 20 Preguntas en la Papa 🌳 (Para Dummies)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/07%20-%20Temas%20Avanzados%20en%20Data%20Analysis/Para%20Dummies/02_Arboles_Decision_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 07 (Árboles de decisión)](../02_Arboles_de_Decision.ipynb). ¿Conoces el juego de las **veinte preguntas**? («¿Es un animal?» «¿Vive en el agua?» ...). Un **árbol de decisión** es eso: una cadena de preguntas de **sí o no** que va **descartando** posibilidades hasta llegar a una respuesta. Aquí lo usaremos para adivinar **qué parcelas de papa se van a enfermar**.

| En el juego de las 20 preguntas | En estadística |
|---|---|
| Cada pregunta de sí/no | Un **corte** del árbol («¿humedad > 60?») |
| La respuesta final | Una **hoja** del árbol |
| Cuántas preguntas encadenas | La **profundidad** (los «pisos») |
| Aprenderte las respuestas de memoria sin entender | El **sobreajuste** |
| Preguntarle a muchos amigos y votar | Un **bosque** de árboles |

Al terminar podrás explicar:
1. Que un árbol se **lee como reglas** («si hay mucha humedad y hace calor, se enferma»).
2. Por qué **más pisos no siempre es mejor**: el árbol puede **memorizar** en vez de aprender.
3. Por qué dos árboles con datos ligeramente distintos pueden dar **reglas distintas**, y por qué **votar entre muchos árboles** ayuda.

> 🔗 Si luego quieres ver **cómo se calcula cada corte, la poda, la importancia de variables y la comparación con intervalos**: [cuaderno estándar](../02_Arboles_de_Decision.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb).

---
## 1. Un Árbol de Dos Preguntas 🌳

Inventamos 400 parcelas: se enferman **cuando hay mucha humedad Y hace calor**, salvo un 8 % de casos «raros» que no siguen la regla (en la vida real siempre hay casos raros). Usamos **280 parcelas para que el árbol aprenda** y dejamos **120 para examinarlo** (parcelas que **nunca vio**). El árbol descubre **solo** las preguntas y los puntos de corte.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text

rng = np.random.default_rng(42)

# 400 parcelas de papa INVENTADAS: se enferman cuando hay mucha humedad Y hace calor (y a veces por mala suerte)
n = 400
humedad = rng.uniform(20, 90, n)
temperatura = rng.uniform(8, 24, n)
enferma = ((humedad > 60) & (temperatura > 16)).astype(int)
flip = rng.random(n) < 0.08                                    # 8 % de casos "raros" que no siguen la regla
enferma = np.where(flip, 1 - enferma, enferma)
parcelas = pd.DataFrame({"humedad": humedad, "temperatura": temperatura, "enferma": enferma})
X, y = parcelas[["humedad", "temperatura"]].values, parcelas["enferma"].values
Xtr, ytr, Xte, yte = X[:280], y[:280], X[280:], y[280:]          # 280 para aprender, 120 para examinar
print("Parcelas enfermas:", f"{y.mean():.0%}")

# Un arbol de 2 pisos: dos preguntas encadenadas
arbol = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr, ytr)
print(export_text(arbol, feature_names=["humedad", "temperatura"]))
print(f"Acierta en {arbol.score(Xtr, ytr):.0%} de las parcelas con las que aprendio y en {arbol.score(Xte, yte):.0%} de las que NO vio")

# Donde decide "enferma" y donde "sana"
g1, g2 = np.meshgrid(np.linspace(20, 90, 200), np.linspace(8, 24, 200))
zona = arbol.predict(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
plt.figure(figsize=(7, 5))
plt.contourf(g1, g2, zona, alpha=0.2, colors=["#16a34a", "#dc2626"], levels=[-0.5, 0.5, 1.5])
plt.scatter(Xtr[:, 0], Xtr[:, 1], c=[["#16a34a", "#dc2626"][v] for v in ytr], s=12)
plt.xlabel("humedad"); plt.ylabel("temperatura"); plt.title("El arbol corta el mapa en rectangulos (rojo = enferma)"); plt.show()


---
**Lección:** el árbol de dos pisos hizo exactamente las preguntas que esperábamos: primero **«¿humedad ≤ 59.9?»** (la regla real era 60) y, si hay mucha humedad, **«¿temperatura ≤ 16.0?»** (la real era 16). Solo cuando hay **mucha humedad y hace calor** dice «enferma». (La rama de humedad baja también pregunta por la temperatura, pero sus dos hojas dicen «sana»: esa pregunta no cambia la respuesta.) Acierta **95 %** de las parcelas con las que aprendió y **93 %** de las que **no vio**: casi igual, buena señal. (Un ≈ 7 % de error es lo esperable: el 8 % de casos raros que metimos no se pueden adivinar.) Fíjate además en el mapa: el árbol corta el terreno en **rectángulos**, porque cada pregunta usa **una sola variable**.

---
## 2. Más Pisos, ¿Más Sabiduría? y el Poder de Votar 🗳️

Probemos árboles con más y más pisos. Con pocos pisos el árbol es **simple**; con muchos, **memoriza** hasta los casos raros. Luego miramos si cambian las reglas cuando cambiamos un poco los datos (**muestras revueltas** de las mismas parcelas) y probamos el truco de **preguntarle a 51 árboles y votar**.

In [ ]:
# Cuantos "pisos" (preguntas encadenadas) le dejamos? Mas pisos = mas memoria, no necesariamente mas sabiduria
filas = []
for pisos in (1, 2, 3, 5, 8, 12):
    m = DecisionTreeClassifier(max_depth=pisos, random_state=0).fit(Xtr, ytr)
    filas.append({"pisos": pisos, "aprendizaje": m.score(Xtr, ytr), "examen": m.score(Xte, yte)})
tabla = pd.DataFrame(filas).set_index("pisos")
print((tabla * 100).round(0).astype(int).astype(str).add("%").to_string())

# Cambian las reglas si cambiamos un poco las parcelas? 5 arboles de 2 pisos, cada uno con una muestra "revuelta" de las mismas parcelas
print("\nPrimera pregunta de 5 arboles distintos (cada uno con una muestra revuelta):")
r = np.random.default_rng(1)
for i in range(5):
    idx = r.choice(len(ytr), len(ytr))
    a = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr[idx], ytr[idx])
    print(f"  arbol {i + 1}: {['humedad', 'temperatura'][a.tree_.feature[0]]} <= {a.tree_.threshold[0]:.1f}")

# "Preguntarle a muchos arboles y votar": un bosque casero con 51 arboles profundos
votos = np.zeros(len(yte))
for i in range(51):
    idx = r.choice(len(ytr), len(ytr))
    votos += DecisionTreeClassifier(random_state=i).fit(Xtr[idx], ytr[idx]).predict(Xte)
acierto_bosque = np.mean((votos > 25) == yte)
acierto_uno = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).score(Xte, yte)
print(f"\nExamen: un arbol profundo = {acierto_uno:.0%} | 51 arboles votando = {acierto_bosque:.0%}")

plt.figure(figsize=(7, 4))
plt.plot(tabla.index, tabla["aprendizaje"] * 100, "o-", color="#dc2626", label="parcelas con las que aprendio")
plt.plot(tabla.index, tabla["examen"] * 100, "o-", color="#2563eb", label="parcelas nuevas (examen)")
plt.xlabel("pisos del arbol"); plt.ylabel("% de aciertos"); plt.legend(); plt.title("Mas pisos: aprende mas, pero el examen no mejora"); plt.show()


---
**Lección:** (1) **Más pisos no mejora el examen:** con **2 pisos** el árbol acierta **95 %** al aprender y **93 %** en el examen; con **8 pisos** llega a **100 %** al aprender (memorizó, incluidos los casos raros) pero el examen **baja a 87 %**. Esa brecha es el **sobreajuste**: como el estudiante que se sabe las respuestas del taller de memoria y falla en el examen. (2) **Un árbol es inestable:** con muestras revueltas, la primera pregunta fue «humedad ≤ 59.5», «≤ 59.8», **«temperatura ≤ 16.2»**, «≤ 59.4» y «≤ 59.9»: casi siempre la humedad, pero en uno de cinco la **primera pregunta cambió de variable**. Pequeños cambios en los datos, reglas distintas. (3) **Votar ayuda:** un árbol profundo acierta **87 %** en el examen y **51 árboles votando**, **92 %**. Como cada árbol se equivoca en casos distintos, al votar los errores se compensan. El costo: el bosque ya **no se lee** como unas pocas reglas.

---
##### 🛠️ Práctica 1: ¿Cuántos pisos?

Con `Xtr`, `ytr` (parcelas para aprender), `Xte`, `yte` (parcelas del examen) y `DecisionTreeClassifier` ya importado:
1. Para cada número de pisos de 1 a 8, entrena un árbol (`max_depth=pisos`, `random_state=0`) y guarda su acierto en el examen en el diccionario `aciertos`.
2. Elige el **menor** número de pisos cuyo acierto sea **a lo más 1 punto menor** (0.01) que el mejor (`pisos_elegidos`).
3. Comprueba con `assert` que esa elección acierta más que un solo piso y que usa **4 pisos o menos**.

In [ ]:
# Escribe tu código aquí

# aciertos = {}
# for pisos in range(1, 9):
#     ...
# mejor = max(aciertos.values())
# pisos_elegidos = ...

# assert aciertos[pisos_elegidos] > aciertos[1] and pisos_elegidos <= 4


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
aciertos = {}
for pisos in range(1, 9):
    m = DecisionTreeClassifier(max_depth=pisos, random_state=0).fit(Xtr, ytr)
    aciertos[pisos] = m.score(Xte, yte)
mejor = max(aciertos.values())
pisos_elegidos = min(p for p, a in aciertos.items() if a >= mejor - 0.01)      # el mas sencillo que casi iguala al mejor
print({p: round(a, 3) for p, a in aciertos.items()}, "-> pisos elegidos =", pisos_elegidos)
assert aciertos[pisos_elegidos] > aciertos[1] and pisos_elegidos <= 4
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué un árbol con **muchos pisos** acierta el 100 % de las parcelas con las que aprendió y aun así falla más en el examen? ¿Con qué situación de un estudiante lo compararías?
2. Si el árbol dice «si humedad > 60 y temperatura > 16, enferma», ¿eso **prueba** que la humedad y el calor **causan** la enfermedad? ¿Qué otra información necesitarías?
3. Un bosque de 51 árboles acertó más que un solo árbol, pero ya no se puede leer como unas pocas reglas. ¿En qué situaciones (por ejemplo, explicarle una decisión a un productor) preferirías un solo árbol pequeño?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>